In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('spam.csv' , encoding='latin-1')[['v1' , 'v2']]
df.columns = ['label' , 'message']

In [4]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
!pip install nltk

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 13.2 MB/s eta 0:00:00


In [10]:
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yashw\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [13]:
def clean_text(text):
   
    text = text.lower()

    
    text = ''.join([c for c in text if c not in string.punctuation])

   
    words = text.split()
    words = [word for word in words if word not in stopwords.words('english')]

    
    return ' '.join(words)


In [14]:
df['cleaned'] = df['message'].apply(clean_text)
df[['message' , 'cleaned']].head()

,message,cleaned
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['cleaned'])
y = df['label']

In [19]:
X.shape

(5572, 9376)

In [20]:
from sklearn.model_selection import train_test_split
X_train , X_test  , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)
X_train.shape , X_test.shape

((4457, 9376), (1115, 9376))

In [21]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()

In [22]:
model.fit(X_train , y_train)

MultinomialNB()

In [23]:
y_pred = model.predict(X_test)

In [24]:
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report
accuracy  = accuracy_score(y_test , y_pred)
conf_matrix = confusion_matrix(y_test , y_pred)
class_report = classification_report(y_test , y_pred)

In [25]:
print("accuracy:" , accuracy)
print("confusion matrix:" , conf_matrix)
print("classification report:" , class_report)

accuracy: 0.9659192825112107
confusion matrix: [[965   0]
 [ 38 112]]
classification report:               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       1.00      0.75      0.85       150

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [26]:
import pickle

In [27]:
pickle.dump(model , open('spam_model.pkl' , 'wb'))
pickle.dump(tfidf , open('vectorizer.pkl' , 'wb'))

In [28]:
def predict_spam(message):
    cleaned_message = clean_text(message)

    transformed_message = tfidf.transform([cleaned_message])

    prediction = model.predict(transformed_message)

    return prediction[0]

In [38]:
new_message = "Congratulations! You won a $1000 gift card."
result = predict_spam(new_message)

if result == "spam":
    print("message is spam")
else:
    print("message is ham")

message is ham


In [37]:
!pip install streamlit
